**Navigation** : [carnet source Python](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Python.ipynb) · [jumeau Rust](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Rust.ipynb)

# Chaînes — Carnet 11 (C#) : Le groupe de Lorentz et la signature un contre trois

Jumeau C# de [Chaînes-11 — le groupe de Lorentz et la signature un contre trois (Python)](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Python.ipynb) : même fil pédagogique, mêmes entrées, même arithmétique exacte — rejouée ici en `BigInteger` et en rationnel exact, BCL seule.

Chapitre 50 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu), sections 54-55. Deux résultats : le **Théorème 54.1** (conditionnel) — le groupe de Lorentz comme isométries de la scission hermitienne — et le **Théorème 55.1**, premier résultat de l'arc au statut de théorème établi : la signature un contre trois vit au choix de l'unité et le transport la porte le long de chaque chaîne sans bouger.

Le patron de la série tient : **Mesuré** ce que nos rationnels exacts rejouent, **Cité** toute la chaîne dérivative du source, **Ouvert** ce qu'il ne prouve pas.


## Le cadre : l'ordre rejoint les rotations

« The rotations fixed, the next chapter joins them to the order coordinate in the Lorentz group » (p. 143). Le chapitre 49 avait dérivé les rotations des directions ; celui-ci y adjoint les boosts — les générateurs qui **mélangent** l'ordre et la direction.

Les statuts diffèrent et le carnet les garde visibles : le 54.1 est **conditionnel** (« conditional on the axiom text and on the results of the chapters named as its premises »), le 55.1 porte la mention « **Standing: theorem** » — la première de la Partie IV sur ce tronçon.


## §54 — Théorème 54.1 (conditionnel) : le groupe de Lorentz comme isométries de la scission hermitienne

« The isometry algebra of the Hermitian split at the qutrit-qubit link [...] is six-dimensional, spanned by the ad matrices of the directions, which fix the unit, and by three boosts generated from the hyperbolic generator by the rotation brackets; at every structure constant it equals the Lorentz algebra of the literature by name, with the Killing form of signature three and three, and the six lift to the qubit end as three anti-Hermitian and three Hermitian generators » (p. 144).

**Notre modèle, déclaré nôtre** : l'espace $\mathbb{R}^4$ de coordonnées $(t, x, y, z)$ muni de $\eta = \mathrm{diag}(4, -1, -1, -1)$ — **la Gram de propagation de l'éq. 81 elle-même** —, trois rotations de bloc spatial (les $\mathrm{ad}$ matrices des directions de la construction, citées), trois boosts dont le bloc $2\times2$ est le générateur littéral imprimé de l'éq. 82. Tout ce qui suit se mesure en `Rationnel` exacte (`BigInteger`) sur ce modèle ; l'égalité « by name » à l'algèbre de la construction reste citée.


In [1]:
// Arithmetique exacte sur Q : Rationnel en BigInteger, matrices 4x4 en tableaux jagges.
// BCL seule : System.Numerics et System.Globalization -- aucun paquet externe, aucun flottant.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Rationnel exact reduit : numerateur et denominateur en BigInteger, denominateur strictement positif.
sealed class Rationnel
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }

    public Rationnel(BigInteger numerateur, BigInteger denominateur)
    {
        if (denominateur == BigInteger.Zero)
            throw new ArgumentException("Denominateur nul");
        if (denominateur < BigInteger.Zero)
        {
            numerateur = -numerateur;
            denominateur = -denominateur;
        }
        BigInteger pgcd = BigInteger.GreatestCommonDivisor(BigInteger.Abs(numerateur), denominateur);
        Numerateur = numerateur / pgcd;
        Denominateur = denominateur / pgcd;
    }

    public static implicit operator Rationnel(int n) => new Rationnel(n, BigInteger.One);
    public static Rationnel Entier(BigInteger n) => new Rationnel(n, BigInteger.One);

    public static Rationnel operator +(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Denominateur + y.Numerateur * x.Denominateur,
                      x.Denominateur * y.Denominateur);
    public static Rationnel operator -(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Denominateur - y.Numerateur * x.Denominateur,
                      x.Denominateur * y.Denominateur);
    public static Rationnel operator -(Rationnel x) => new Rationnel(-x.Numerateur, x.Denominateur);
    public static Rationnel operator *(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Numerateur, x.Denominateur * y.Denominateur);
    public static Rationnel operator /(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Denominateur, x.Denominateur * y.Numerateur);

    public static bool operator ==(Rationnel x, Rationnel y) =>
        x.Numerateur == y.Numerateur && x.Denominateur == y.Denominateur;
    public static bool operator !=(Rationnel x, Rationnel y) =>
        !(x == y);
    public static bool operator <(Rationnel x, Rationnel y) =>
        x.Numerateur * y.Denominateur < y.Numerateur * x.Denominateur;
    public static bool operator >(Rationnel x, Rationnel y) =>
        x.Numerateur * y.Denominateur > y.Numerateur * x.Denominateur;

    public override bool Equals(object obj) => obj is Rationnel r && this == r;
    public override int GetHashCode() =>
        Numerateur.GetHashCode() ^ (Denominateur.GetHashCode() * 31);
    public override string ToString() =>
        Denominateur == BigInteger.One ? Numerateur.ToString() : $"{Numerateur}/{Denominateur}";

    public Rationnel Abs() => new Rationnel(BigInteger.Abs(Numerateur), Denominateur);
    public bool EstNul => Numerateur == BigInteger.Zero;
}

// Operations matricielles exactes (dimension quelconque : 4x4 ici, 2x2 au plan).
static class Outils
{
    public static Rationnel[][] Mul(Rationnel[][] A, Rationnel[][] B) =>
        Enumerable.Range(0, A.Length)
            .Select(i => Enumerable.Range(0, B[0].Length)
                .Select(j => Enumerable.Range(0, A.Length)
                    .Aggregate((Rationnel)0, (acc, k) => acc + A[i][k] * B[k][j]))
                .ToArray())
            .ToArray();
    public static Rationnel[][] Add(Rationnel[][] A, Rationnel[][] B) =>
        Enumerable.Range(0, A.Length)
            .Select(i => Enumerable.Range(0, A[0].Length).Select(j => A[i][j] + B[i][j]).ToArray())
            .ToArray();
    public static Rationnel[][] Sub(Rationnel[][] A, Rationnel[][] B) =>
        Enumerable.Range(0, A.Length)
            .Select(i => Enumerable.Range(0, A[0].Length).Select(j => A[i][j] - B[i][j]).ToArray())
            .ToArray();
    public static Rationnel[][] Br(Rationnel[][] A, Rationnel[][] B) => Sub(Mul(A, B), Mul(B, A));
    public static bool EqM(Rationnel[][] A, Rationnel[][] B) =>
        Enumerable.Range(0, A.Length).All(i =>
            Enumerable.Range(0, A[0].Length).All(j => A[i][j] == B[i][j]));
    public static Rationnel[][] Transp(Rationnel[][] A) =>
        Enumerable.Range(0, A.Length)
            .Select(i => Enumerable.Range(0, A.Length).Select(j => A[j][i]).ToArray())
            .ToArray();
    public static Rationnel[][] Neg(Rationnel[][] A) =>
        A.Select(row => row.Select(x => -x).ToArray()).ToArray();
    public static Rationnel[][] Zeros(int n)
    {
        var M = new Rationnel[n][];
        for (int i = 0; i < n; i++)
        {
            M[i] = new Rationnel[n];
            for (int j = 0; j < n; j++)
                M[i][j] = 0;
        }
        return M;
    }
    public static Rationnel[][] E(int i, int j)
    {
        var M = Zeros(4);
        M[i][j] = 1;
        return M;
    }
    public static bool Nulle(Rationnel[][] A) =>
        A.SelectMany(row => row).All(v => v.EstNul);
}

// L'appariement : eta = diag(4, -1, -1, -1) -- la Gram de propagation de l'eq. 81.
var eta = new Rationnel[][] {
    new Rationnel[] { 4, 0, 0, 0 },
    new Rationnel[] { 0, -1, 0, 0 },
    new Rationnel[] { 0, 0, -1, 0 },
    new Rationnel[] { 0, 0, 0, -1 }
};

// Trois rotations (blocs spatiaux, (Ji)_jk = -eps_ijk) et trois boosts (plans (1/2, 2)).
var J1 = Outils.Sub(Outils.E(3, 2), Outils.E(2, 3));
var J2 = Outils.Sub(Outils.E(1, 3), Outils.E(3, 1));
var J3 = Outils.Sub(Outils.E(2, 1), Outils.E(1, 2));
Rationnel[][] Boost(int i)
{
    var K = Outils.Zeros(4);
    K[0][i] = new Rationnel(1, 2);
    K[i][0] = 2;
    return K;
}
var K1 = Boost(1);
var K2 = Boost(2);
var K3 = Boost(3);
var laBase = new[] { J1, J2, J3, K1, K2, K3 };
var noms = new[] { "J1", "J2", "J3", "K1", "K2", "K3" };

// Condition d'algebre : X^T eta + eta X = 0 pour les six.
bool Residu(Rationnel[][] X, Rationnel[][] P)
{
    var R = Outils.Add(Outils.Mul(Outils.Transp(X), P), Outils.Mul(P, X));
    return Outils.Nulle(R);
}
var resids = noms.Zip(laBase, (n, X) => (n, ok: Residu(X, eta))).ToDictionary(kv => kv.n, kv => kv.ok);
Console.WriteLine("X^T eta + eta X = 0 pour les six generateurs -> {" +
    string.Join(", ", resids.Select(kv => $"'{kv.Key}': {kv.Value}")) + "} -> " +
    resids.Values.All(v => v));


X^T eta + eta X = 0 pour les six generateurs -> {'J1': True, 'J2': True, 'J3': True, 'K1': True, 'K2': True, 'K3': True} -> True


### Lecture du résultat (isométries)

**Mesuré** : les six générateurs — trois rotations qui fixent l'axe du temps (les « ad matrices of the directions » de la construction, citées) et trois boosts — satisfont la condition d'algèbre d'isométrie de $\eta$. Le bloc $2\times2$ de chaque boost, $\begin{psmallmatrix} 0 & 1/2 \\ 2 & 0 \end{psmallmatrix}$, est le générateur imprimé de l'éq. 82 : notre modèle et la construction partagent les mêmes matrices de plan.


In [2]:
// Les crochets de l'eq. 80 : [J1,J2] = J3, [Ji,Kj] = eps_ijk Kk, [K1,K2] = -J3.
using System;
using System.Linq;

int Eps(int i, int j, int k)
{
    if (i == j || j == k || i == k)
        return 0;
    if ((i, j, k) == (1, 2, 3) || (i, j, k) == (2, 3, 1) || (i, j, k) == (3, 1, 2))
        return 1;
    return -1;
}

var rotRot = new[] {
    Outils.EqM(Outils.Br(J1, J2), J3),
    Outils.EqM(Outils.Br(J2, J3), J1),
    Outils.EqM(Outils.Br(J3, J1), J2)
};
Console.WriteLine("[J1,J2]=J3, [J2,J3]=J1, [J3,J1]=J2 -> [" + string.Join(", ", rotRot) + "]");

Rationnel[][] Echelle(Rationnel[][] M, Rationnel c) =>
    M.Select(row => row.Select(x => x * c).ToArray()).ToArray();
bool melangeOk = true;
for (int i = 1; i <= 3; i++)
{
    for (int j = 1; j <= 3; j++)
    {
        var attendu = Outils.Zeros(4);
        for (int k = 1; k <= 3; k++)
        {
            if (Eps(i, j, k) != 0)
                attendu = Outils.Add(attendu, Echelle(laBase[3 + k - 1], Eps(i, j, k)));
        }
        if (!Outils.EqM(Outils.Br(laBase[i - 1], laBase[3 + j - 1]), attendu))
            melangeOk = false;
    }
}
Console.WriteLine("[Ji,Kj] = eps_ijk Kk (les 9 crochets) -> " + melangeOk);
var boostBoost = new[] {
    Outils.EqM(Outils.Br(K1, K2), Outils.Neg(J3)),
    Outils.EqM(Outils.Br(K2, K3), Outils.Neg(J1)),
    Outils.EqM(Outils.Br(K3, K1), Outils.Neg(J2))
};
Console.WriteLine("[K1,K2]=-J3, [K2,K3]=-J1, [K3,K1]=-J2 -> [" + string.Join(", ", boostBoost) + "]");


[J1,J2]=J3, [J2,J3]=J1, [J3,J1]=J2 -> [True, True, True]


[Ji,Kj] = eps_ijk Kk (les 9 crochets) -> True


[K1,K2]=-J3, [K2,K3]=-J1, [K3,K1]=-J2 -> [True, True, True]


### Lecture du résultat (constantes de structure)

**Mesuré** : l'éq. 80 exacte sur notre modèle — les rotations se ferment sur elles-mêmes, les crochets rotation-boost **font tourner le boost** (les neuf $[J_i, K_j] = \epsilon_{ijk} K_k$), et deux boosts donnent **moins une rotation**. « at every structure constant it equals the Lorentz algebra of the literature by name » : ici ce sont nos propres rationnels exacts qui font le compte.


In [3]:
// L'identite de Jacobi sur les 216 triples de la base.
using System;
using System.Linq;

bool jacobi = !laBase.Any(X => laBase.Any(Y => laBase.Any(Z =>
    !Outils.Nulle(Outils.Add(
        Outils.Add(Outils.Br(Outils.Br(X, Y), Z), Outils.Br(Outils.Br(Y, Z), X)),
        Outils.Br(Outils.Br(Z, X), Y))))));
Console.WriteLine("Jacobi 216/216 triples -> " + jacobi);


Jacobi 216/216 triples -> True


### Lecture du résultat

**Mesuré** : l'identité de Jacobi tient sur les 216 triples de la base — le crochet est une structure d'algèbre de Lie, pas seulement quinze égalités isolées.


In [4]:
// Forme de Killing k(X,Y) = Tr(ad_X ad_Y) dans la base (J1..J3, K1..K3).
using System;
using System.Collections.Generic;
using System.Linq;

static class Killing
{
    // Signature par elimination symetrique exacte : pivot de plus grand module, complement de Schur.
    public static (int, int) SignatureSym(Rationnel[][] S)
    {
        var M = S.Select(row => row.ToArray()).ToArray();
        var signes = new List<int>();
        while (true)
        {
            int pr = -1;
            var best = Rationnel.Entier(-1);
            for (int r = 0; r < M.Length; r++)
                if (M[r][r].Abs() > best)
                {
                    best = M[r][r].Abs();
                    pr = r;
                }
            if (pr < 0 || M[pr][pr].EstNul)
                break;
            signes.Add(M[pr][pr] > 0 ? 1 : -1);
            var p = M[pr][pr];
            var garde = Enumerable.Range(0, M.Length).Where(i => i != pr).ToArray();
            M = garde.Select(i => garde
                .Select(j => M[i][j] - M[i][pr] * M[pr][j] / p).ToArray()).ToArray();
        }
        return (signes.Count(s => s == 1), signes.Count(s => s == -1));
    }
}

int[] Coeffs(Rationnel[][] C)
{
    if (Outils.Nulle(C))
        return new int[6];
    for (int k = 0; k < 6; k++)
    {
        if (Outils.EqM(C, laBase[k]))
        {
            var v = new int[6]; v[k] = 1; return v;
        }
        if (Outils.EqM(C, Outils.Neg(laBase[k])))
        {
            var v = new int[6]; v[k] = -1; return v;
        }
    }
    return null;
}
var ads = new List<int[][]>();
bool fermeture = true;
foreach (var X in laBase)
{
    var cols = new List<int[]>();
    foreach (var e in laBase)
    {
        var v = Coeffs(Outils.Br(X, e));
        if (v is null)
        {
            fermeture = false;
            v = new int[6];
        }
        cols.Add(v);
    }
    ads.Add(cols.ToArray());
}
var kappa = new Rationnel[6][];
for (int a = 0; a < 6; a++)
{
    kappa[a] = new Rationnel[6];
    for (int b = 0; b < 6; b++)
    {
        Rationnel s = 0;
        for (int j = 0; j < 6; j++)
            for (int k = 0; k < 6; k++)
                s = s + ads[a][j][k] * ads[b][k][j];
        kappa[a][b] = s;
    }
}
Console.WriteLine("fermeture de la base -> " + fermeture);
Console.WriteLine("kappa diagonale (J1..J3, K1..K3) : [" +
    string.Join(", ", Enumerable.Range(0, 6).Select(i => kappa[i][i].ToString())) + "]");
Console.WriteLine("kappa hors-diagonale nulle -> " +
    Enumerable.Range(0, 6).All(i => Enumerable.Range(0, 6).Where(j => j != i)
        .All(j => kappa[i][j].EstNul)));

var sigKappa = Killing.SignatureSym(kappa);
bool kappaDiagOk = Enumerable.Range(0, 6)
    .All(i => kappa[i][i] == (i < 3 ? Rationnel.Entier(-4) : Rationnel.Entier(4)));
Console.WriteLine("signature de kappa (pivots symetriques) : (" + sigKappa.Item1 + ", " +
    sigKappa.Item2 + ") (attendu (3, 3)) ; diagonale (-4,-4,-4,4,4,4) : " + kappaDiagOk);


fermeture de la base -> True


kappa diagonale (J1..J3, K1..K3) : [-4, -4, -4, 4, 4, 4]


kappa hors-diagonale nulle -> True


signature de kappa (pivots symetriques) : (3, 3) (attendu (3, 3)) ; diagonale (-4,-4,-4,4,4,4) : True


### Lecture du résultat (Killing)

**Mesuré** : $\kappa(J,J) = -4$ pour chaque rotation, $\kappa(K,K) = +4$ pour chaque boost, tous les termes croisés nuls — la forme de Killing est $\mathrm{diag}(-4,-4,-4,\,4,4,4)$, signature **(3, 3)** par élimination symétrique exacte. Les sous-espaces se séparent d'eux-mêmes : les rotations portent (0, 3), les boosts (3, 0) — et la moitié (0, 3) est précisément la lecture du contre-test du source (« with the unit struck the six-dimensional algebra moves to the three-dimensional isometry algebra of the directions alone at the signature zero and three », p. 145, cité).


## Les Grams des deux liens : l'éq. 79 et l'éq. 81

À l'unité hermitienne du lien qutrit-qubit (dimension six) : « the unit squaring to the link dimension six, every direction step to minus three halves and every interference term zero » (p. 145) — l'éq. 79, $G = \tfrac32\,\mathrm{diag}(4,-1,-1,-1)$. À la chaîne série (2,2), la Gram de propagation (éq. 81) est $\mathrm{diag}(4,-1,-1,-1)$, signature $(1,3)$, le rapport $-4$.

**Le rapport est le même aux deux types de lien** : $6 \div \tfrac32 = 4 \div 1 = 4$ — l'unité pèse quatre directions, quel que soit le lien. Cité : la Gram calculée cellule par cellule aux **six types de paire**, la même scission partout.


In [5]:
// Eq. 79 : G = diag(6, -3/2, -3/2, -3/2) = (3/2)*diag(4,-1,-1,-1) ; eq. 81 : diag(4,-1,-1,-1).
using System;
using System.Collections.Generic;
using System.Linq;

static class Grams
{
    // Premier moteur du source : les signes des pivots (diagonale deja).
    public static (int, int) SigPivots(Rationnel[] diag)
    {
        var s = diag.Select(v => v > 0 ? 1 : -1).ToList();
        return (s.Count(x => x == 1), s.Count(x => x == -1));
    }
    // Second moteur : le critere de Sylvester (mineurs principaux).
    public static (int, int) SigSylvester(Rationnel[] diag)
    {
        var prods = new List<Rationnel>();
        Rationnel p = 1;
        foreach (var v in diag)
        {
            p = p * v;
            prods.Add(p);
        }
        var suite = new List<int> { 1 };
        suite.AddRange(prods.Select(x => x > 0 ? 1 : -1));
        int negatifs = 0;
        for (int a = 0; a + 1 < suite.Count; a++)
            if (suite[a] != suite[a + 1])
                negatifs++;
        return (diag.Length - negatifs, negatifs);
    }
}

var G79 = new Rationnel[] { 6, new Rationnel(-3, 2), new Rationnel(-3, 2), new Rationnel(-3, 2) };
var G81 = new Rationnel[] { 4, -1, -1, -1 };
var c32 = new Rationnel(3, 2);
Console.WriteLine("(3/2)*diag(4,-1,-1,-1) = eq.79 -> " +
    Enumerable.Range(0, 4).All(i => c32 * G81[i] == G79[i]));
var ratio79 = G79[0] / G79[1];
var ratio81 = G81[0] / G81[1];
Console.WriteLine("ratio eq.79 : 6 / (-3/2) = " + ratio79 + " ; ratio eq.81 : 4 / (-1) = " + ratio81);

foreach (var (nom, G) in new[] { ("eq.79", G79), ("eq.81", G81) })
{
    var sp = Grams.SigPivots(G);
    var ss = Grams.SigSylvester(G);
    Console.WriteLine($"{nom} : signes pivots ({sp.Item1}, {sp.Item2}) ; Sylvester ({ss.Item1}, {ss.Item2})");
}
var sig79 = Grams.SigPivots(G79);
var sig81 = Grams.SigPivots(G81);
var syl79 = Grams.SigSylvester(G79);
var syl81 = Grams.SigSylvester(G81);


(3/2)*diag(4,-1,-1,-1) = eq.79 -> True


ratio eq.79 : 6 / (-3/2) = -4 ; ratio eq.81 : 4 / (-1) = -4


eq.79 : signes pivots (1, 3) ; Sylvester (1, 3)


eq.81 : signes pivots (1, 3) ; Sylvester (1, 3)


### Lecture du résultat (Grams)

**Mesuré** : les deux Grams donnent la signature (1, 3) par les **deux méthodes des deux moteurs du source** — les signes des pivots (« the pivot signs at the first engine ») et les mineurs principaux (« the leading principal minors at the second », critère de Sylvester) — et le rapport $-4$ est commun aux deux liens. **Cité** : le calcul cellule par cellule, l'égalité à la rangée (2,2) aux deux bouts, 116/116 pas (§55).


## §55 — Théorème 55.1 : un contre trois à chaque pas de la chaîne

« At every chain and step carrying a (2,2) link, the Gram of the propagation reads one against three, the boost of the chain is its isometry with the mixing block nonzero, and the one-unit rotation [...] is the isometry of a definite plane and of no restriction of that Gram » (p. 146). **Standing: theorem.**

Le census (cité) : 34/36 états de chaîne parcourus à 116 pas sur 82 liens (2,2) série des neuf toiles enregistrées — l'électron, le muon, le tau, le proton, le neutron jusqu'au deutéron, le mot photon, la toile univers, le puits neutron et le puits tau. Nous, nous mesurons les **matrices imprimées** du plan : les deux générateurs, leurs deux plans, leurs carrés et leurs résidus exacts.


In [6]:
// Le plan (eq. 82 et figure 28) : l'unite hermitienne et une direction portee.
using System;
using System.Linq;

var PL = new Rationnel[][] { new Rationnel[] { 4, 0 }, new Rationnel[] { 0, -1 } };          // le plan 4, 0 ; 0, -1 (indefini)
var Dp = new Rationnel[][] { new Rationnel[] { 1, 0 }, new Rationnel[] { 0, new Rationnel(1, 4) } }; // le plan des ordres 1, 0 ; 0, 1/4 (defini)
var Kp = new Rationnel[][] { new Rationnel[] { 0, new Rationnel(1, 2) }, new Rationnel[] { 2, 0 } }; // le boost 0, 1/2 ; 2, 0
var Rp = new Rationnel[][] { new Rationnel[] { 0, new Rationnel(-1, 2) }, new Rationnel[] { 2, 0 } }; // la rotation d'unite 0, -1/2 ; 2, 0
var I2 = new Rationnel[][] { new Rationnel[] { 1, 0 }, new Rationnel[] { 0, 1 } };
var Z2 = new Rationnel[][] { new Rationnel[] { 0, 0 }, new Rationnel[] { 0, 0 } };

Rationnel[][] Resid2(Rationnel[][] A, Rationnel[][] P) =>
    Outils.Add(Outils.Mul(Outils.Transp(A), P), Outils.Mul(P, A));
string M2(Rationnel[][] M) =>
    "[" + string.Join("; ", M.Select(row => string.Join(", ", row.Select(v => v.ToString())))) + "]";
var rBoost = Resid2(Kp, PL);
bool boostOk = Outils.EqM(rBoost, Z2);
Console.WriteLine("K^T PL + PL K = " + M2(rBoost) + " -> nul : " + boostOk);
bool kCarre = Outils.EqM(Outils.Mul(Kp, Kp), I2);
Console.WriteLine("K^2 = I -> " + kCarre);
bool rCarre = Outils.EqM(Outils.Mul(Rp, Rp), Outils.Neg(I2));
Console.WriteLine("R^2 = -I -> " + rCarre);
var rEucl = Resid2(Rp, Dp);
bool rEuclOk = Outils.EqM(rEucl, Z2);
Console.WriteLine("R^T D + D R = " + M2(rEucl) + " -> nul (euclidien) : " + rEuclOk);
var rGram = Resid2(Rp, PL);
bool rGramOk = Outils.EqM(rGram, new Rationnel[][] { new Rationnel[] { 0, -4 }, new Rationnel[] { -4, 0 } });
Console.WriteLine("R^T PL + PL R = " + M2(rGram) + " -> attendu [[0, -4], [-4, 0]] : " + rGramOk +
    "; non nul : pas une isometrie de la Gram");


K^T PL + PL K = [0, 0; 0, 0] -> nul : True


K^2 = I -> True


R^2 = -I -> True


R^T D + D R = [0, 0; 0, 0] -> nul (euclidien) : True


R^T PL + PL R = [0, -4; -4, 0] -> attendu [[0, -4], [-4, 0]] : True; non nul : pas une isometrie de la Gram


### Lecture du résultat (le plan, ligne à ligne de la figure 28)

**Mesuré** : le boost $\begin{psmallmatrix} 0 & 1/2 \\ 2 & 0 \end{psmallmatrix}$ a un résidu nul sur $\mathrm{diag}(4,-1)$ et **carré l'identité** ; la rotation d'unité $\begin{psmallmatrix} 0 & -1/2 \\ 2 & 0 \end{psmallmatrix}$ a **carré moins l'identité**, résidu nul sur le plan défini $\mathrm{diag}(1, \tfrac14)$ (l'appariement euclidien), et sur la Gram restreinte son résidu est exactement $\begin{psmallmatrix} 0 & -4 \\ -4 & 0 \end{psmallmatrix}$ — non nul : **pas une isométrie**. La table de la figure 28 (p. 150), reproduite colonne par colonne : générateur, plan, carré, appariement préservé, résidu, lettre — Euclidien pour la rotation, Lorentzien pour le boost, aucune isométrie pour la restriction.


In [7]:
// Le releve au bout qubit : le triple litteral du ch. 55 (deja croise au ch. 49).
using System;
using System.Linq;

// Complexe exact : couple (partie reelle, partie imaginaire) de Rationnel ; matrices 2x2 jaggees.
(Rationnel, Rationnel) CScale((Rationnel, Rationnel) z, Rationnel n) => (z.Item1 * n, z.Item2 * n);
(Rationnel, Rationnel) CNeg((Rationnel, Rationnel) z) => (-z.Item1, -z.Item2);

var C0 = ((Rationnel)0, (Rationnel)0);
var CI = ((Rationnel)0, (Rationnel)1);
var G1 = new (Rationnel, Rationnel)[][] {
    new[] { C0, CScale(CI, new Rationnel(-1, 2)) },
    new[] { CScale(CI, new Rationnel(-1, 2)), C0 }
};
var G2 = new (Rationnel, Rationnel)[][] {
    new[] { C0, (new Rationnel(1, 2), (Rationnel)0) },
    new[] { (new Rationnel(-1, 2), (Rationnel)0), C0 }
};
var G3 = new (Rationnel, Rationnel)[][] {
    new[] { CScale(CI, new Rationnel(-1, 2)), C0 },
    new[] { C0, CScale(CI, new Rationnel(1, 2)) }
};
var T = new[] { G1, G2.Select(row => row.Select(z => CNeg(z)).ToArray()).ToArray(), G3 }; // re-orientation (flip G2)

(Rationnel, Rationnel)[][] Dagger((Rationnel, Rationnel)[][] M)
{
    (Rationnel, Rationnel) Conj((Rationnel, Rationnel) z) => (z.Item1, -z.Item2);
    return new[]
    {
        new[] { Conj(M[0][0]), Conj(M[1][0]) },
        new[] { Conj(M[0][1]), Conj(M[1][1]) }
    };
}
bool MEq2((Rationnel, Rationnel)[][] A, (Rationnel, Rationnel)[][] B) =>
    A[0][0] == B[0][0] && A[0][1] == B[0][1] && A[1][0] == B[1][0] && A[1][1] == B[1][1];
(Rationnel, Rationnel)[][] NegC((Rationnel, Rationnel)[][] M) =>
    M.Select(row => row.Select(z => CNeg(z)).ToArray()).ToArray();
var anti = T.Select(t => MEq2(Dagger(t), NegC(t))).ToList();
Console.WriteLine("les trois rotations anti-hermitiennes (dagger = -M) : [" +
    string.Join(", ", anti) + "] -> " + anti.All(v => v));
Console.WriteLine("les trois boosts hermitiens au bout qubit : CITE -- le releve est celui de la");
Console.WriteLine("construction ; notre generateur de plan porte l'isometrie eta, mesurée plus haut");


les trois rotations anti-hermitiennes (dagger = -M) : [True, True, True] -> True


les trois boosts hermitiens au bout qubit : CITE -- le releve est celui de la


construction ; notre generateur de plan porte l'isometrie eta, mesurée plus haut


### Lecture du résultat (le relevé)

**Mesuré** : les trois générateurs de rotation du triple littéral — celui-là même que la chaîne transporte (« the transposed triple », p. 147) — sont anti-hermitiens au dagger ordinaire : les « three anti-Hermitian » du théorème. **Cité** : les « three Hermitian generators » du côté boost — le relevé du bout qubit appartient à la construction, notre modèle porte l'isométrie $\eta$.


In [8]:
// Le cone nul du plan 4, 0 ; 0, -1, et le contraste Pell (rapport non carre).
using System;

bool coneOk = true;
foreach (var v in new[] { (1, 2), (1, -2) })
{
    Rationnel t = v.Item1;
    Rationnel x = v.Item2;
    var q = PL[0][0] * t * t + PL[1][1] * x * x;
    if (!q.EstNul)
        coneOk = false;
    Console.WriteLine("direction (" + v.Item1 + ", " + v.Item2 + ") : 4t^2 - x^2 = " + q);
}
int pell = 3 * 3 - 2 * 2 * 2;
Console.WriteLine("Pell : 3^2 - 2*2^2 = " + pell +
    " -- la solution fondamentale de t^2 - 2x^2 = 1 (Chaines-09)");


direction (1, 2) : 4t^2 - x^2 = 0


direction (1, -2) : 4t^2 - x^2 = 0


Pell : 3^2 - 2*2^2 = 1 -- la solution fondamentale de t^2 - 2x^2 = 1 (Chaines-09)


### Lecture du résultat (le cône et Pell)

**Mesuré** : le plan de l'unité porte exactement deux directions nulles, $(1, 2)$ et $(1, -2)$ — le cône dessiné au-dessus de la chaîne (figure 27) ; et $3^2 - 2\cdot2^2 = 1$, la solution fondamentale de Pell. **Carrefour** : la famille de Pell donne une **infinité** de cartes au rapport non carré (Chaînes-09), quand le rapport carré donne un groupe **fini** — les 24 rotations du réseau de Chaînes-10. « while the Euclidean pairing admits a finite group at every ratio » (p. 145, cité) : nos deux carnets précédents en sont les deux moitiés.


## Ce que le census porte (§54, cité)

6 objets énumérés ; la Gram calculée cellule par cellule aux **six types de paire**, la scission (1, 3) partout ; le contre-test de l'unité frappée — l'algèbre de dimension six passe à l'algèbre de directions de dimension trois à la signature (0, 3), « with no plane of the unit » ; cinq contre-tests aux lettres épinglées ; treize contrôles lus pareil aux deux moteurs ; la reproduction par une seconde construction, égale à chaque cellule ; la plus longue chaîne de prémisses : douze chapitres.


In [9]:
// Les comptes du census (§54), cites tels qu'imprimes (p. 145-146).
using System;

var census54 = new (string Libelle, string Valeur)[] {
    ("objets enumeres", "6"),
    ("types de paire (Gram cellule par cellule)", "6"),
    ("contre-tests aux lettres epinglees", "5"),
    ("controles lus pareil aux deux moteurs", "13"),
    ("longueur de la plus longue chaine de premices (chapitres)", "12"),
};
foreach (var (k, v) in census54)
    Console.WriteLine($"{k} : {v} (CITE)");
Console.WriteLine("contre-test nomme : unite frappee -> algebre de directions seule, dimension 3,");
Console.WriteLine("signature (0, 3), sans plan de l'unite (cite) ; notre moitie negative de kappa la lit");


objets enumeres : 6 (CITE)


types de paire (Gram cellule par cellule) : 6 (CITE)


contre-tests aux lettres epinglees : 5 (CITE)


controles lus pareil aux deux moteurs : 13 (CITE)


longueur de la plus longue chaine de premices (chapitres) : 12 (CITE)


contre-test nomme : unite frappee -> algebre de directions seule, dimension 3,


signature (0, 3), sans plan de l'unite (cite) ; notre moitie negative de kappa la lit


## Ce que le census porte (§55, cité)

Un contre trois à **116/116** pas (défini à aucun, autre à aucun) ; le boost isométrie à 116/116 ; la rotation d'unité isométrie de son plan défini à 116/116 et **d'aucune restriction de la Gram** ; l'orientation de la triple portée **préservée aux pas pairs, renversée aux impairs** (50/116) — figure 26 : a1 p, x1 r, x2 p, x3 r, a2 p sur la chaîne de l'électron ; les constantes de structure **niées aux pas impairs** à la base littérale portée (78/90 au premier site inséré, les zéros tenant) et égales à la table des crochets à **90/90** à la base ré-orientée à chaque pas — c'est le flip de Chaînes-10 qui alterne avec la parité ; le pôle à chaque lien, fonction à deux points $(30+30\sqrt2)/2304$, 204/204 ; le résultat de boost ouvert transcrit intégralement et re-dérivé au cadre du lien (2,3), les deux comptes imprimés par la cellule de census qui suit.

Quatre contre-tests : l'unité frappée → le plan devient défini et le type rotation à chaque pas ; l'orientation renversée à chaque pas → les comptes bougent, la signature pas ; le lien (2,3) terminal du proton et du photon lu comme lien de chaîne → la Gram passe à $\mathrm{diag}(6,-1,-1,-1)$ à ces deux pas, signature inchangée ; la porte frappée → toutes les cellules absentes.


In [10]:
// Les comptes du census (§55), cites tels qu'imprimes (p. 146-149).
using System;

var census55 = new (string Libelle, string Valeur)[] {
    ("etats de chaine parcourus", "34 / 36"),
    ("pas marches", "116"),
    ("liens (2,2) serie", "82"),
    ("un contre trois", "116 / 116 (defini a aucun, autre a aucun)"),
    ("boost isometrie", "116 / 116"),
    ("rotation d'unite sur son plan defini", "116 / 116"),
    ("orientation preservee (pairs) / renversee (impairs)", "50 / 116"),
    ("constantes egales a la table (base re-orientee)", "90 / 90 a 116/116"),
    ("pole a deux points", "204 / 204"),
    ("resultat de boost ouvert transcrit / re-derive", "15 / 15 et 10 / 10"),
};
foreach (var (k, v) in census55)
    Console.WriteLine($"{k} : {v} (CITE)");
var lettres = new[] { "a1 p", "x1 r", "x2 p", "x3 r", "a2 p" };
Console.WriteLine("orientation de la chaine de l'electron (figure 26, citee) : [" +
    string.Join(", ", lettres.Select(l => $"'{l}'")) + "]");


etats de chaine parcourus : 34 / 36 (CITE)


pas marches : 116 (CITE)


liens (2,2) serie : 82 (CITE)


un contre trois : 116 / 116 (defini a aucun, autre a aucun) (CITE)


boost isometrie : 116 / 116 (CITE)


rotation d'unite sur son plan defini : 116 / 116 (CITE)


orientation preservee (pairs) / renversee (impairs) : 50 / 116 (CITE)


constantes egales a la table (base re-orientee) : 90 / 90 a 116/116 (CITE)


pole a deux points : 204 / 204 (CITE)


resultat de boost ouvert transcrit / re-derive : 15 / 15 et 10 / 10 (CITE)


orientation de la chaine de l'electron (figure 26, citee) : ['a1 p', 'x1 r', 'x2 p', 'x3 r', 'a2 p']


## Ouvert — ligne à ligne

1. **L'unité de boost ouverte au ch. 56** : « the one unit open at chapter 56 » — le prochain carnet (Théorème 56.2, open result, et la famille de Pell à l'unité debout).
2. **Le résultat de boost ouvert** dont le 55.1 transcrit les cellules : ses prémisses (l'ouvert 92.1, les théorèmes 92.2 et 115.1) ne sont pas prouvées ici.
3. **Le pôle** : la fonction à deux points $(30+30\sqrt2)/2304$ citée telle quelle — non rejouée.
4. **La rangée d'insertion série** et **la rangée à deux points** : prémisses ouvertes citées du ch. 10.
5. **Les huit rapports carrés de tick** (§54) : l'élément d'ordre infini existe à chacun (Pell) ; le compte des cartes par rapport n'est pas rejoué ici.


### Exercice 1 — l'exponentielle du boost au point hyperbolique rationnel

$K^2 = I$ (mesuré ci-dessus) donne $\exp(\theta K) = \cosh\theta\, I + \sinh\theta\, K$. Au point pythagoricien $(3,4,5)$ : $\cosh = 5/4$, $\sinh = 3/4$. Construire $M$, vérifier la condition **de groupe** $M^T P_L M = P_L$ — quadratique, à distinguer du résidu linéaire d'algèbre — puis lire $\tanh = \sinh/\cosh$ et le rapport $x/t$ de la première colonne.


In [11]:
// Exercice 1 : stub C.1 -- le carnet s'execute de bout en bout meme non complete.
using System;

var ch = new Rationnel(5, 4);  // cosh au point pythagoricien (3,4,5)
var sh = new Rationnel(3, 4);  // sinh au meme point
Rationnel[][] M = null;  // TODO etudiant : ch*I2 + sh*Kp, puis Outils.EqM(Outils.Mul(Outils.Mul(Outils.Transp(M), PL), M), PL),
                         // puis tanh = sh/ch et le rapport M[1][0]/M[0][0]
Console.WriteLine("Exercice a completer");


Exercice a completer


### Exercice 2 — les signatures des deux moitiés

La forme de Killing mesurée est diagonale par blocs. Extraire sa restriction au sous-espace des rotations $(J_1, J_2, J_3)$ et à celui des boosts $(K_1, K_2, K_3)$ : deux signatures attendues, (0, 3) et (3, 0) — l'algèbre de directions du contre-test, et son complément.


In [12]:
// Exercice 2 : stub C.1 -- les signatures des restrictions de Killing, exactement comme au source.
using System;

(int, int)? sigRot = null;   // TODO etudiant : Killing.SignatureSym sur la sous-matrice 3x3 des J
(int, int)? sigBoost = null; // TODO etudiant : idem sur celle des K
Console.WriteLine("Exercice a completer");


Exercice a completer


### Exercice 3 — le lien (2,3) lu comme lien de chaîne

Le troisième contre-test : lire le lien terminal (2,3) du proton et du photon comme un lien de chaîne déplace la Gram vers $\mathrm{diag}(6, -1, -1, -1)$ — l'entrée de l'unité passe à la dimension du lien. Calculer les signes pivots et Sylvester de cette Gram : la signature bouge-t-elle ? Et son rapport unité/direction ?


In [13]:
// Exercice 3 : stub C.1 -- pivots ET Sylvester, les deux moteurs du source.
using System;

var G23 = new Rationnel[] { 6, -1, -1, -1 };
(int, int)? sig23 = null;  // TODO etudiant : Grams.SigPivots(G23) et Grams.SigSylvester(G23), puis le ratio 6/(-1)
Console.WriteLine("Exercice a completer");


Exercice a completer


## Conclusion

Le groupe de Lorentz n'est pas plus posé que le groupe des rotations : les six générateurs tombent de la scission hermitienne, les constantes (80) sont exactes, la Killing sépare trois et trois, et le rapport $-4$ — l'unité contre quatre directions — tient aux deux types de lien. La signature un contre trois vit au choix de l'unité, et le transport la porte le long de chaque chaîne sans bouger (théorème 55.1) : la rotation d'unité reste euclidienne, le boost reste lorentzien, à chaque pas.

La suite (ch. 51) place le boost à l'unité : l'ouvert 56.2 et la famille de Pell à l'unité debout — grain 12.


In [14]:
// Releve final : les checks du carnet.
using System;
using System.Linq;

var checks = new (string Libelle, bool Ok)[] {
    ("six residus d'isometrie (eta) nuls", resids.Values.All(v => v)),
    ("rot-rot [J1,J2]=J3 (x3)", rotRot.All(v => v)),
    ("melange [Ji,Kj]=eps_ijk Kk (x9)", melangeOk),
    ("boost-boost [K1,K2]=-J3 (x3)", boostBoost.All(v => v)),
    ("Jacobi 216/216 triples", jacobi),
    ("Killing diag(-4,-4,-4,4,4,4), sig (3,3)", kappaDiagOk && sigKappa == (3, 3)),
    ("(3/2)*diag(4,..) = eq.79", Enumerable.Range(0, 4).All(i => c32 * G81[i] == G79[i])),
    ("ratio -4 aux deux liens", ratio79 == Rationnel.Entier(-4) && ratio81 == Rationnel.Entier(-4)),
    ("signatures (1,3) par pivots ET Sylvester",
        sig79 == (1, 3) && sig81 == (1, 3) && syl79 == (1, 3) && syl81 == (1, 3)),
    ("plan : residu boost nul, K^2=I, R^2=-I", boostOk && kCarre && rCarre),
    ("R euclidien (residu nul) et non-Gram ([[0,-4],[-4,0]])", rEuclOk && rGramOk),
    ("rotations anti-hermitiennes (x3)", anti.All(v => v)),
    ("cone nul (1,2) et (1,-2)", coneOk),
    ("Pell 3^2 - 2*2^2 = 1", pell == 1),
};
int nbOk = checks.Count(c => c.Ok);
foreach (var (k, v) in checks)
    Console.WriteLine($"{k} -> {v}");
Console.WriteLine($"Total : {nbOk} / {checks.Length}");


six residus d'isometrie (eta) nuls -> True


rot-rot [J1,J2]=J3 (x3) -> True


melange [Ji,Kj]=eps_ijk Kk (x9) -> True


boost-boost [K1,K2]=-J3 (x3) -> True


Jacobi 216/216 triples -> True


Killing diag(-4,-4,-4,4,4,4), sig (3,3) -> True


... (5 outputs left out)

rotations anti-hermitiennes (x3) -> True


cone nul (1,2) et (1,-2) -> True


Pell 3^2 - 2*2^2 = 1 -> True


Total : 14 / 14


## À retenir

1. **Le groupe de Lorentz se dérive, comme les rotations** : la même scission hermitienne de la
   trace donne six générateurs — trois $J$ anti-hermitiens (les rotations), trois $K$ hermitiens
   (les boosts) — et les six résidus d'isométrie sont nuls.
2. **Les constantes de structure (80) sont exactes** : $[J_1,J_2] = J_3$, les neuf mélanges
   $[J_i, K_j] = \epsilon_{ijk} K_k$, $[K_1, K_2] = -J_3$ — et l'identité de Jacobi tient sur les
   $216$ triples de la base.
3. **La forme de Killing sépare trois et trois** : $\kappa = \mathrm{diag}(-4,-4,-4,\,4,4,4)$,
   signature $(3,3)$ par élimination symétrique exacte — les rotations portent $(0,3)$, les boosts
   $(3,0)$, et c'est la lecture du contre-test du source.
4. **Un contre trois aux deux liens** : $G_{79} = \tfrac32\,\mathrm{diag}(4,-1,-1,-1)$ et
   $G_{81} = \mathrm{diag}(4,-1,-1,-1)$, tous deux de signature $(1,3)$ par pivots **et** par
   Sylvester, avec le même rapport $-4$ — l'unité contre les quatre directions.
5. **La signature vit au choix de l'unité, et le transport la porte** : la rotation d'unité reste
   euclidienne ($R^2 = -I$, résidu nul, et sa « Gram » $[[0,-4],[-4,0]]$ n'en est pas une), le
   boost reste lorentzien ($K^2 = I$), le cône nul porte $(1,2)$ et $(1,-2)$, et
   $3^2 - 2 \cdot 2^2 = 1$ est la famille de Pell à l'unité debout.


## Vérifiez votre compréhension

1. Pourquoi les rotations sont-elles anti-hermitiennes et les boosts hermitiens, et que mesure ce
   partage ?
2. Que vaut la forme de Killing sur cette algèbre, et comment s'y lit la signature $(3,3)$ ?
3. Les deux Grams de l'éq. 79 et de l'éq. 81 n'ont pas les mêmes coefficients : pourquoi donnent-ils
   pourtant la même signature ?
4. Qu'est-ce qui distingue le plan de la rotation d'unité de celui du boost, et où le théorème 55.1
   intervient-il ?

<details>
<summary>Réponses</summary>

1. Elles tombent de la scission hermitienne : $J = i\sigma/2$ est anti-hermitien, $K = \sigma/2$
   est hermitien. Le partage est exactement ce que la forme de Killing mesure — $\kappa(J,J) = -4$,
   $\kappa(K,K) = +4$, tous les croisés nuls.
2. $\kappa = \mathrm{diag}(-4,-4,-4,\,4,4,4)$ : trois négatifs (les rotations) et trois positifs
   (les boosts), d'où la signature $(3,3)$ — et, séparément, $(0,3)$ sur les rotations et $(3,0)$
   sur les boosts.
3. Ils ne diffèrent que par le facteur scalaire $3/2$ : $G_{79} = \tfrac32 G_{81}$. Un facteur
   positif ne change ni le signe ni le nombre des pivots, donc les deux formes ont la même
   signature $(1,3)$ — et le rapport des deux vaut $-4$ dans les deux cas.
4. La rotation d'unité vérifie $R^2 = -I$ — elle est euclidienne, de résidu nul, et sa forme
   associée $[[0,-4],[-4,0]]$ n'est pas une Gram (non définie) ; le boost vérifie $K^2 = I$ et son
   cône nul porte exactement $(1,2)$ et $(1,-2)$. Le théorème 55.1 est le transport : la signature
   un-contre-trois ne bouge pas le long de la chaîne.

</details>


## Pour aller plus loin

*Chaînes-12 — les boosts et l'unité une* est le
carnet que la conclusion annonce : le boost à l'unité, l'ouvert 56.2 et la famille de Pell à l'unité
debout.

En amont, [Chaînes-10 — le groupe des rotations](Chaines-10-Groupe-Rotations-Automorphismes-Directions-Python.ipynb)
est le carnet dont la conclusion annonçait celui-ci (« the next chapter joins them to the order
coordinate in the Lorentz group »), et [Chaînes-09 — la coordonnée d'ordre](Chaines-09-Coordonnee-Ordre-Chaine-Serie-Python.ipynb)
fournit la coordonnée que le théorème 55.1 transporte le long de la chaîne.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
reste le contraste de méthode : là où une conservation se dérive d'une symétrie continue, les six
générateurs de ce carnet sont **dérivés d'une scission**, et c'est la forme de Killing — non un
postulat — qui sépare les trois rotations des trois boosts.
